# 4 · Exporting the models for the browser

The slide decks run the models themselves, in the viewer's browser, with onnxruntime-web, as in the VPT talk. This
notebook shows the export (`export_onnx.py` does it for all models) and the check that the browser runs exactly
what was trained.

One ONNX graph of the frozen base model serves three methods. Besides the image it takes both kinds of task file:
VPT prompts (B, 12, P, 768), inserted before every block, and LoRA pairs A (B, 12, 2, R, 768) and B (B, 12, 2, 768, R)
beside the query and value weights of every block, with α / r folded into B.

* no prompts and zero pairs: the plain base model (the linear probe's features)
* prompts only: VPT-deep
* pairs only: LoRA

Because the task files are inputs, every image of a batch could even carry a different task. LoRA runs *unmerged*
here, on purpose: merged, it would be a separate 172 MB model per task, like full fine-tuning. The fully fine-tuned
models and the VPT prompt files are the VPT talk's, loaded from its repository.

In [ ]:
import sys, json, time, math
from pathlib import Path
ROOT = Path.cwd().parent            # demo/lora
VPT = ROOT.parent / "vpt"           # data, linear-probe and full fine-tuning runs
sys.path.insert(0, str(ROOT))
import numpy as np, torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

In [ ]:
from export_onnx import Encoder, export
from lora_model import LoRAViT
torch.set_grad_enabled(False)
tmp = ROOT / "web" / "notebook_check"; tmp.mkdir(parents=True, exist_ok=True)
vit = LoRAViT(2, rank=0).vit.eval()
t0 = time.time()
sess = export(vit, tmp / "enc.fp16.onnx")            # also prints the check against PyTorch on random inputs
print(f"exported and checked in {time.time() - t0:.0f}s: {(tmp / 'enc.fp16.onnx').stat().st_size / 1e6:.0f} MB")
enc = Encoder(vit).eval()

## Does the browser model compute the same thing?

Cosine similarity between the PyTorch features and the fp16 ONNX features on 16 DermaMNIST test images: with nothing
fed in, with the real VPT-deep prompts, and with the real LoRA pairs.

In [ ]:
from safetensors.torch import load_file
from lora_model import load_task
x = np.load(VPT / "data" / "dermamnist_224" / "test_images.npy", mmap_mode="r")[:16]
img = ((np.asarray(x).astype(np.float32) / 255 - 0.5) / 0.5).transpose(0, 3, 1, 2).copy()
P = load_file(VPT / "weights" / "dermamnist_vpt-deep-p50_full.safetensors")["prompts"].numpy()
A, B = (z.numpy() for z in load_task(load_file(ROOT / "weights" / "dermamnist_lora-r8_full.safetensors"), 7, pretrained=False).stacked())
rep = lambda a: np.broadcast_to(a, (16, *a.shape)).copy()
none_p, za, zb = np.zeros((16, 12, 0, 768), np.float32), np.zeros((16, 12, 2, 1, 768), np.float32), np.zeros((16, 12, 2, 768, 1), np.float32)
for name, pr, pa, pb in [("nothing (linear probe)", none_p, za, zb), ("VPT-deep prompts", rep(P), za, zb), ("LoRA pairs, rank 8", none_p, rep(A), rep(B))]:
    ref = enc(*(torch.from_numpy(v) for v in (img, pr, pa, pb))).numpy()
    got = sess.run(None, {"image": img, "prompts": pr, "lora_a": pa, "lora_b": pb})[0]
    cos = (ref * got).sum(1) / np.linalg.norm(ref, axis=1) / np.linalg.norm(got, axis=1)
    print(f"{name:24s} cosine to PyTorch: min {cos.min():.6f}")

## Where the decks get the models

`upload_models.py` publishes `web/` to the free model repository
[valinor61/lora-medmnist](https://huggingface.co/valinor61/lora-medmnist). The decks fetch `peft_encoder.fp16.onnx`
and the task's 1.2 MB LoRA file from there, and the VPT prompt file and the fully fine-tuned model from
[valinor61/vpt-medmnist](https://huggingface.co/valinor61/vpt-medmnist), on first use. They keep them in the browser's
Cache Storage (shared with the VPT deck) and run them with onnxruntime-web. `heads.json` holds every classification
head and the test metrics of exactly these ONNX models:

In [ ]:
heads = json.loads((ROOT / "web" / "heads.json").read_text())
for t, e in heads["tasks"].items():
    for key in ("methods_1k", "methods"):
        print(f"{t:15s} {key:10s}", {m: (v["test_torch"]["bal_acc"], v["test_web"]["bal_acc"]) for m, v in e[key].items()}, "(PyTorch, ONNX balanced accuracy)")
del sess
import shutil; shutil.rmtree(tmp, ignore_errors=True)   # the check files are not needed